# Naive Bayes Model

In [2]:
import pandas as pd

In [3]:
df_train = pd.read_csv(r'C:\Users\prchandr\Downloads\playground-series-s6e4 (1)\train.csv')
df_test = pd.read_csv(r'C:\Users\prchandr\Downloads\playground-series-s6e4 (1)\test.csv')

X = df_train.drop('Irrigation_Need', axis=1)
y = df_train['Irrigation_Need'].copy()

In [4]:
df_train.head(5)

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


# Gaussian Naive Bayes

In [18]:
# ============================================
# GAUSSIAN NAIVE BAYES
# THRESHOLD ANALYSIS
# USE ONLY KAGGLE TRAIN DATA
# Target:
#   Irrigation_Need
# ============================================


# ============================================
# 1. IMPORTS
# ============================================

import numpy as np
import pandas as pd
import warnings

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")


# ============================================
# 2. BASIC SETUP
# ============================================

TARGET = "Irrigation_Need"
RANDOM_STATE = 42

df_train.columns = df_train.columns.str.strip()


# ============================================
# 3. SPLIT KAGGLE TRAIN DATA INTO FEATURES + TARGET
# ============================================

X_full = df_train.drop(columns=[TARGET]).copy()
y_full = df_train[TARGET].copy()

target_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

inverse_target_mapping = {
    0: "Low",
    1: "Medium",
    2: "High"
}

y_full_enc = y_full.map(target_mapping)

print("Target mapping:")
for k, v in target_mapping.items():
    print(f"{k} -> {v}")


# ============================================
# 4. TRAIN / TEST SPLIT USING ONLY KAGGLE TRAIN DATA
# ============================================

X_train, X_test, y_train_enc, y_test_enc = train_test_split(
    X_full,
    y_full_enc,
    test_size=0.20,
    stratify=y_full_enc,
    random_state=RANDOM_STATE
)

print("\nTraining shape:", X_train.shape)
print("Testing shape:", X_test.shape)


# ============================================
# 5. IDENTIFY COLUMN TYPES
# ============================================

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("\nNumeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


# ============================================
# 6. PREPROCESSING
# GaussianNB works naturally with continuous features
# One-hot encoded categorical variables are included
# Output must be dense for GaussianNB
# ============================================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)


# ============================================
# 7. GAUSSIAN NAIVE BAYES PIPELINE
# ============================================

final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GaussianNB())
])


# ============================================
# 8. FIT MODEL
# ============================================

final_model.fit(X_train, y_train_enc)


# ============================================
# 9. GENERATE PREDICTED PROBABILITIES
# ============================================

y_test_pred_proba = final_model.predict_proba(X_test)

test_proba_df = pd.DataFrame(
    y_test_pred_proba,
    columns=["Prob_Low", "Prob_Medium", "Prob_High"]
)

print("\nPredicted probabilities:")
print(test_proba_df.head())


# ============================================
# 10. BASELINE PREDICTIONS
# Default rule = highest probability
# ============================================

y_test_baseline_pred_enc = final_model.predict(X_test)

print("\nBASELINE METRICS")
print("Accuracy:", accuracy_score(y_test_enc, y_test_baseline_pred_enc))
print("Macro F1:", f1_score(y_test_enc, y_test_baseline_pred_enc, average="macro"))

print("\nBaseline Classification Report:")
print(classification_report(
    y_test_enc,
    y_test_baseline_pred_enc,
    target_names=["Low", "Medium", "High"]
))

print("Baseline Confusion Matrix:")
print(confusion_matrix(y_test_enc, y_test_baseline_pred_enc))


# ============================================
# 11. CHOOSE ONE CLASS + ONE METRIC
# Focus class = High
# Metric = Recall for High
# ============================================

FOCUS_CLASS_NAME = "High"
FOCUS_CLASS_ENC = target_mapping[FOCUS_CLASS_NAME]

baseline_focus_recall = recall_score(
    y_test_enc,
    y_test_baseline_pred_enc,
    labels=[FOCUS_CLASS_ENC],
    average=None
)[0]

baseline_focus_f1 = f1_score(
    y_test_enc,
    y_test_baseline_pred_enc,
    labels=[FOCUS_CLASS_ENC],
    average=None
)[0]

print(f"\nFocus class: {FOCUS_CLASS_NAME}")
print(f"Baseline Recall for {FOCUS_CLASS_NAME}: {baseline_focus_recall:.4f}")
print(f"Baseline F1 for {FOCUS_CLASS_NAME}: {baseline_focus_f1:.4f}")


# ============================================
# 12. APPLY THRESHOLD TO CHOSEN CLASS
# If Prob_High >= threshold -> predict High
# Else -> keep baseline prediction
# ============================================

HIGH_THRESHOLD = 0.45

high_probs = y_test_pred_proba[:, FOCUS_CLASS_ENC]

y_test_threshold_pred_enc = y_test_baseline_pred_enc.copy()
y_test_threshold_pred_enc[high_probs >= HIGH_THRESHOLD] = FOCUS_CLASS_ENC

print(f"\nApplied threshold for class '{FOCUS_CLASS_NAME}': {HIGH_THRESHOLD}")


# ============================================
# 13. EVALUATE THRESHOLDED PREDICTIONS
# ============================================

threshold_focus_recall = recall_score(
    y_test_enc,
    y_test_threshold_pred_enc,
    labels=[FOCUS_CLASS_ENC],
    average=None
)[0]

threshold_focus_f1 = f1_score(
    y_test_enc,
    y_test_threshold_pred_enc,
    labels=[FOCUS_CLASS_ENC],
    average=None
)[0]

print("\nTHRESHOLDED METRICS")
print("Accuracy:", accuracy_score(y_test_enc, y_test_threshold_pred_enc))
print("Macro F1:", f1_score(y_test_enc, y_test_threshold_pred_enc, average="macro"))
print(f"Recall for {FOCUS_CLASS_NAME}: {threshold_focus_recall:.4f}")
print(f"F1 for {FOCUS_CLASS_NAME}: {threshold_focus_f1:.4f}")

print("\nThresholded Classification Report:")
print(classification_report(
    y_test_enc,
    y_test_threshold_pred_enc,
    target_names=["Low", "Medium", "High"]
))

print("Thresholded Confusion Matrix:")
print(confusion_matrix(y_test_enc, y_test_threshold_pred_enc))


# ============================================
# 14. COMPARISON TABLE
# ============================================

comparison_df = pd.DataFrame({
    "Model Version": ["Baseline", "Thresholded"],
    "Accuracy": [
        accuracy_score(y_test_enc, y_test_baseline_pred_enc),
        accuracy_score(y_test_enc, y_test_threshold_pred_enc)
    ],
    "Macro_F1": [
        f1_score(y_test_enc, y_test_baseline_pred_enc, average="macro"),
        f1_score(y_test_enc, y_test_threshold_pred_enc, average="macro")
    ],
    f"{FOCUS_CLASS_NAME}_Recall": [
        baseline_focus_recall,
        threshold_focus_recall
    ],
    f"{FOCUS_CLASS_NAME}_F1": [
        baseline_focus_f1,
        threshold_focus_f1
    ]
})

print("\nComparison Table:")
print(comparison_df)

Target mapping:
Low -> 0
Medium -> 1
High -> 2

Training shape: (504000, 20)
Testing shape: (126000, 20)

Numeric columns: ['id', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
Categorical columns: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

Predicted probabilities:
   Prob_Low  Prob_Medium  Prob_High
0  0.194912     0.684598   0.120490
1  0.778539     0.221352   0.000109
2  0.514081     0.479238   0.006682
3  0.377817     0.566561   0.055621
4  0.646956     0.346398   0.006645

BASELINE METRICS
Accuracy: 0.7451587301587301
Macro F1: 0.5975271441309615

Baseline Classification Report:
              precision    recall  f1-score   support

         Low       0.77      0.88      0.82     73983
      Medium       0.70      0.58      0.63     47815
        High  

# Multinomial Naive Bayes

In [ ]:
# ============================================
# FAST MULTINOMIAL NAIVE BAYES
# NO SCALING
# NO RANDOMIZEDSEARCHCV
# WITH THRESHOLD ANALYSIS
# Uses:
#   df_train -> labeled data with target column
#   df_test  -> unlabeled data for final scoring
# Target:
#   Irrigation_Need
# ============================================


# ============================================
# 1. IMPORTS
# ============================================

import numpy as np
import pandas as pd
import warnings

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")


# ============================================
# 2. BASIC SETUP
# ============================================

TARGET = "Irrigation_Need"
RANDOM_STATE = 42

df_train.columns = df_train.columns.str.strip()
df_test.columns = df_test.columns.str.strip()


# ============================================
# 3. SPLIT df_train INTO FEATURES + TARGET
# ============================================

X_full = df_train.drop(columns=[TARGET]).copy()
y_full = df_train[TARGET].copy()

target_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

inverse_target_mapping = {
    0: "Low",
    1: "Medium",
    2: "High"
}

y_full_enc = y_full.map(target_mapping)

print("Target mapping:")
for k, v in target_mapping.items():
    print(f"{k} -> {v}")

X_train, X_valid, y_train_enc, y_valid_enc = train_test_split(
    X_full,
    y_full_enc,
    test_size=0.20,
    stratify=y_full_enc,
    random_state=RANDOM_STATE
)


# ============================================
# 4. IDENTIFY COLUMN TYPES
# ============================================

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("\nNumeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


# ============================================
# 5. PREPROCESSING
# ============================================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)


# ============================================
# 6. FINAL MODEL
# Fixed hyperparameters for speed and stability
# ============================================

final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", MultinomialNB(alpha=1.0, fit_prior=True))
])


# ============================================
# 7. FIT MODEL ON TRAINING SPLIT
# ============================================

final_model.fit(X_train, y_train_enc)


# ============================================
# 8. STEP 1: GENERATE PREDICTED PROBABILITIES
# ============================================

y_valid_pred_proba = final_model.predict_proba(X_valid)

valid_proba_df = pd.DataFrame(
    y_valid_pred_proba,
    columns=["Prob_Low", "Prob_Medium", "Prob_High"]
)

print("\nPredicted probabilities on validation set:")
print(valid_proba_df.head())


# ============================================
# 9. STEP 2: BASELINE PREDICTIONS
# Default rule = choose class with highest probability
# ============================================

y_valid_baseline_pred_enc = final_model.predict(X_valid)

print("\nBASELINE METRICS")
print("Accuracy:", accuracy_score(y_valid_enc, y_valid_baseline_pred_enc))
print("Macro F1:", f1_score(y_valid_enc, y_valid_baseline_pred_enc, average="macro"))

print("\nBaseline Classification Report:")
print(classification_report(
    y_valid_enc,
    y_valid_baseline_pred_enc,
    target_names=["Low", "Medium", "High"]
))

print("Baseline Confusion Matrix:")
print(confusion_matrix(y_valid_enc, y_valid_baseline_pred_enc))


# ============================================
# 10. STEP 3: CHOOSE ONE CLASS + ONE METRIC
# Focus class = High
# Metric = Recall for High
# ============================================

FOCUS_CLASS_NAME = "High"
FOCUS_CLASS_ENC = target_mapping[FOCUS_CLASS_NAME]

baseline_focus_recall = recall_score(
    y_valid_enc,
    y_valid_baseline_pred_enc,
    labels=[FOCUS_CLASS_ENC],
    average=None
)[0]

baseline_focus_f1 = f1_score(
    y_valid_enc,
    y_valid_baseline_pred_enc,
    labels=[FOCUS_CLASS_ENC],
    average=None
)[0]

print(f"\nFocus class: {FOCUS_CLASS_NAME}")
print(f"Baseline Recall for {FOCUS_CLASS_NAME}: {baseline_focus_recall:.4f}")
print(f"Baseline F1 for {FOCUS_CLASS_NAME}: {baseline_focus_f1:.4f}")


# ============================================
# 11. STEP 4: APPLY A THRESHOLD TO THE FOCUS CLASS
# If Prob_High >= threshold -> predict High
# Else -> keep baseline prediction
# ============================================

HIGH_THRESHOLD = 0.45

high_probs = y_valid_pred_proba[:, FOCUS_CLASS_ENC]

y_valid_threshold_pred_enc = y_valid_baseline_pred_enc.copy()
y_valid_threshold_pred_enc[high_probs >= HIGH_THRESHOLD] = FOCUS_CLASS_ENC

print(f"\nApplied threshold for class '{FOCUS_CLASS_NAME}': {HIGH_THRESHOLD}")


# ============================================
# 12. EVALUATE THRESHOLDED PREDICTIONS
# ============================================

threshold_focus_recall = recall_score(
    y_valid_enc,
    y_valid_threshold_pred_enc,
    labels=[FOCUS_CLASS_ENC],
    average=None
)[0]

threshold_focus_f1 = f1_score(
    y_valid_enc,
    y_valid_threshold_pred_enc,
    labels=[FOCUS_CLASS_ENC],
    average=None
)[0]

print("\nTHRESHOLDED METRICS")
print("Accuracy:", accuracy_score(y_valid_enc, y_valid_threshold_pred_enc))
print("Macro F1:", f1_score(y_valid_enc, y_valid_threshold_pred_enc, average="macro"))
print(f"Recall for {FOCUS_CLASS_NAME}: {threshold_focus_recall:.4f}")
print(f"F1 for {FOCUS_CLASS_NAME}: {threshold_focus_f1:.4f}")

print("\nThresholded Classification Report:")
print(classification_report(
    y_valid_enc,
    y_valid_threshold_pred_enc,
    target_names=["Low", "Medium", "High"]
))

print("Thresholded Confusion Matrix:")
print(confusion_matrix(y_valid_enc, y_valid_threshold_pred_enc))


# ============================================
# 13. COMPARISON TABLE
# ============================================

comparison_df = pd.DataFrame({
    "Model Version": ["Baseline", "Thresholded"],
    "Accuracy": [
        accuracy_score(y_valid_enc, y_valid_baseline_pred_enc),
        accuracy_score(y_valid_enc, y_valid_threshold_pred_enc)
    ],
    "Macro_F1": [
        f1_score(y_valid_enc, y_valid_baseline_pred_enc, average="macro"),
        f1_score(y_valid_enc, y_valid_threshold_pred_enc, average="macro")
    ],
    f"{FOCUS_CLASS_NAME}_Recall": [
        baseline_focus_recall,
        threshold_focus_recall
    ],
    f"{FOCUS_CLASS_NAME}_F1": [
        baseline_focus_f1,
        threshold_focus_f1
    ]
})

print("\nComparison Table:")
print(comparison_df)


# ============================================
# 14. REFIT ON ALL OF df_train
# ============================================

final_model.fit(X_full, y_full_enc)


# ============================================
# 15. SCORE UNLABELED df_test
# ============================================

df_test_pred_proba = final_model.predict_proba(df_test)

df_test_baseline_pred_enc = final_model.predict(df_test)
df_test_baseline_pred = pd.Series(df_test_baseline_pred_enc).map(inverse_target_mapping)

df_test_threshold_pred_enc = df_test_baseline_pred_enc.copy()
df_test_threshold_pred_enc[df_test_pred_proba[:, FOCUS_CLASS_ENC] >= HIGH_THRESHOLD] = FOCUS_CLASS_ENC
df_test_threshold_pred = pd.Series(df_test_threshold_pred_enc).map(inverse_target_mapping)

df_test_results = df_test.copy()
df_test_results["Predicted_Irrigation_Need_Baseline"] = df_test_baseline_pred.values
df_test_results["Predicted_Irrigation_Need_Thresholded"] = df_test_threshold_pred.values
df_test_results["Prob_Low"] = df_test_pred_proba[:, 0]
df_test_results["Prob_Medium"] = df_test_pred_proba[:, 1]
df_test_results["Prob_High"] = df_test_pred_proba[:, 2]

print("\nSCORED df_test SAMPLE")
print(df_test_results.head())


Target mapping:
Low -> 0
Medium -> 1
High -> 2

Numeric columns: ['id', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
Categorical columns: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

Predicted probabilities on validation set:
        Prob_Low    Prob_Medium      Prob_High
0   9.997718e-01   2.282026e-04  4.196249e-135
1   1.000000e+00   1.990199e-33   0.000000e+00
2   8.225147e-06   9.999918e-01   8.818481e-14
3   1.532880e-04   9.998467e-01   1.279573e-66
4  1.956039e-146  3.263368e-128   1.000000e+00

BASELINE METRICS
Accuracy: 0.3947063492063492
Macro F1: 0.30103800521613866

Baseline Classification Report:
              precision    recall  f1-score   support

         Low       0.64      0.55      0.59     73983
      Medium       0.47      0.14      0.

In [17]:
df_test_results.to_csv("multinomial_nb_threshold_predictions.csv", index=False)

# Comparing both models

In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, classification_report, confusion_matrix

FOCUS_CLASS_NAME = "High"
FOCUS_CLASS_ENC = target_mapping[FOCUS_CLASS_NAME]

# The most recent Gaussian code used:
# y_test_enc
# y_test_baseline_pred_enc
# y_test_threshold_pred_enc
#
# The most updated final Multinomial code used:
# y_valid_enc
# y_valid_baseline_pred_enc
# y_valid_threshold_pred_enc
#
# These should come from the same df_train split configuration.
# This check makes sure the holdout labels match before comparing models.

if len(y_test_enc) != len(y_valid_enc):
    raise ValueError("Gaussian and Multinomial holdout sets are different sizes, so the results are not directly comparable.")

if not np.array_equal(np.asarray(y_test_enc), np.asarray(y_valid_enc)):
    raise ValueError("Gaussian and Multinomial are not using the same holdout labels. Re-run both models on the same split before comparing.")

y_true = np.asarray(y_test_enc)

def get_class_metric(metric_func, y_true, y_pred, class_id):
    return metric_func(y_true, y_pred, labels=[class_id], average=None, zero_division=0)[0]

comparison_df = pd.DataFrame([
    {
        "Model": "GaussianNB",
        "Version": "Baseline",
        "Accuracy": accuracy_score(y_true, y_test_baseline_pred_enc),
        "Macro_F1": f1_score(y_true, y_test_baseline_pred_enc, average="macro"),
        "Weighted_F1": f1_score(y_true, y_test_baseline_pred_enc, average="weighted"),
        f"{FOCUS_CLASS_NAME}_Precision": get_class_metric(precision_score, y_true, y_test_baseline_pred_enc, FOCUS_CLASS_ENC),
        f"{FOCUS_CLASS_NAME}_Recall": get_class_metric(recall_score, y_true, y_test_baseline_pred_enc, FOCUS_CLASS_ENC),
        f"{FOCUS_CLASS_NAME}_F1": get_class_metric(f1_score, y_true, y_test_baseline_pred_enc, FOCUS_CLASS_ENC),
    },
    {
        "Model": "GaussianNB",
        "Version": "Thresholded",
        "Accuracy": accuracy_score(y_true, y_test_threshold_pred_enc),
        "Macro_F1": f1_score(y_true, y_test_threshold_pred_enc, average="macro"),
        "Weighted_F1": f1_score(y_true, y_test_threshold_pred_enc, average="weighted"),
        f"{FOCUS_CLASS_NAME}_Precision": get_class_metric(precision_score, y_true, y_test_threshold_pred_enc, FOCUS_CLASS_ENC),
        f"{FOCUS_CLASS_NAME}_Recall": get_class_metric(recall_score, y_true, y_test_threshold_pred_enc, FOCUS_CLASS_ENC),
        f"{FOCUS_CLASS_NAME}_F1": get_class_metric(f1_score, y_true, y_test_threshold_pred_enc, FOCUS_CLASS_ENC),
    },
    {
        "Model": "MultinomialNB",
        "Version": "Baseline",
        "Accuracy": accuracy_score(y_true, y_valid_baseline_pred_enc),
        "Macro_F1": f1_score(y_true, y_valid_baseline_pred_enc, average="macro"),
        "Weighted_F1": f1_score(y_true, y_valid_baseline_pred_enc, average="weighted"),
        f"{FOCUS_CLASS_NAME}_Precision": get_class_metric(precision_score, y_true, y_valid_baseline_pred_enc, FOCUS_CLASS_ENC),
        f"{FOCUS_CLASS_NAME}_Recall": get_class_metric(recall_score, y_true, y_valid_baseline_pred_enc, FOCUS_CLASS_ENC),
        f"{FOCUS_CLASS_NAME}_F1": get_class_metric(f1_score, y_true, y_valid_baseline_pred_enc, FOCUS_CLASS_ENC),
    },
    {
        "Model": "MultinomialNB",
        "Version": "Thresholded",
        "Accuracy": accuracy_score(y_true, y_valid_threshold_pred_enc),
        "Macro_F1": f1_score(y_true, y_valid_threshold_pred_enc, average="macro"),
        "Weighted_F1": f1_score(y_true, y_valid_threshold_pred_enc, average="weighted"),
        f"{FOCUS_CLASS_NAME}_Precision": get_class_metric(precision_score, y_true, y_valid_threshold_pred_enc, FOCUS_CLASS_ENC),
        f"{FOCUS_CLASS_NAME}_Recall": get_class_metric(recall_score, y_true, y_valid_threshold_pred_enc, FOCUS_CLASS_ENC),
        f"{FOCUS_CLASS_NAME}_F1": get_class_metric(f1_score, y_true, y_valid_threshold_pred_enc, FOCUS_CLASS_ENC),
    }
])

print("MODEL COMPARISON TABLE")
print(comparison_df.sort_values(by=["Model", "Version"]).round(4))

print("\nGAUSSIAN NB - BASELINE CLASSIFICATION REPORT")
print(classification_report(
    y_true,
    y_test_baseline_pred_enc,
    target_names=["Low", "Medium", "High"],
    zero_division=0
))

print("GAUSSIAN NB - THRESHOLDED CLASSIFICATION REPORT")
print(classification_report(
    y_true,
    y_test_threshold_pred_enc,
    target_names=["Low", "Medium", "High"],
    zero_division=0
))

print("MULTINOMIAL NB - BASELINE CLASSIFICATION REPORT")
print(classification_report(
    y_true,
    y_valid_baseline_pred_enc,
    target_names=["Low", "Medium", "High"],
    zero_division=0
))

print("MULTINOMIAL NB - THRESHOLDED CLASSIFICATION REPORT")
print(classification_report(
    y_true,
    y_valid_threshold_pred_enc,
    target_names=["Low", "Medium", "High"],
    zero_division=0
))

print("GAUSSIAN NB - BASELINE CONFUSION MATRIX")
print(confusion_matrix(y_true, y_test_baseline_pred_enc))

print("GAUSSIAN NB - THRESHOLDED CONFUSION MATRIX")
print(confusion_matrix(y_true, y_test_threshold_pred_enc))

print("MULTINOMIAL NB - BASELINE CONFUSION MATRIX")
print(confusion_matrix(y_true, y_valid_baseline_pred_enc))

print("MULTINOMIAL NB - THRESHOLDED CONFUSION MATRIX")
print(confusion_matrix(y_true, y_valid_threshold_pred_enc))

best_macro_f1_row = comparison_df.loc[comparison_df["Macro_F1"].idxmax()]
best_high_recall_row = comparison_df.loc[comparison_df[f"{FOCUS_CLASS_NAME}_Recall"].idxmax()]
best_high_f1_row = comparison_df.loc[comparison_df[f"{FOCUS_CLASS_NAME}_F1"].idxmax()]

print("\nBEST BY MACRO F1")
print(best_macro_f1_row)

print(f"\nBEST BY {FOCUS_CLASS_NAME} RECALL")
print(best_high_recall_row)

print(f"\nBEST BY {FOCUS_CLASS_NAME} F1")
print(best_high_f1_row)

MODEL COMPARISON TABLE
           Model      Version  Accuracy  Macro_F1  Weighted_F1  \
0     GaussianNB     Baseline    0.7452    0.5975       0.7331   
1     GaussianNB  Thresholded    0.7452    0.5992       0.7332   
2  MultinomialNB     Baseline    0.3947    0.3010       0.4319   
3  MultinomialNB  Thresholded    0.3946    0.3010       0.4318   

   High_Precision  High_Recall  High_F1  
0          0.8028       0.2151   0.3393  
1          0.7920       0.2201   0.3445  
2          0.0515       0.5871   0.0947  
3          0.0515       0.5871   0.0947  

GAUSSIAN NB - BASELINE CLASSIFICATION REPORT
              precision    recall  f1-score   support

         Low       0.77      0.88      0.82     73983
      Medium       0.70      0.58      0.63     47815
        High       0.80      0.22      0.34      4202

    accuracy                           0.75    126000
   macro avg       0.76      0.56      0.60    126000
weighted avg       0.74      0.75      0.73    126000

GAUSSIAN 

## Discussion

I focused on the High irrigation need class because it is the most important class that needs improvement. Failing to identify a field that actually requires high irrigation (a false negative) can lead to under-watering, crop stress, and reduced yield. In contrast, predicting High irrigation when it is not actually needed (a false positive) may result in some excess water usage, but it is generally less harmful than missing a critical irrigation need. Therefore I used recall as my main measure of the model performance.

It is better to overestimate irrigation needs and apply slightly more water than necessary than to underestimate and risk damaging crops. 

I applied a probability threshold of 0.45 to the High class. This means that any observation with a predicted probability of at least 0.45 for High irrigation need is classified as High, even if it is not the highest-probability class under the default rule.

Compared to other models, I really liked the Gaussian Naive Bayes model because it worked well for how fast and easy it was to build. It predicted irrigation needs most of the time and doing a good job across all categories. However, when it came to fields that truly needed a lot of water, it only identified a small portion of them, meaning it missed many important cases. After applying the threshold, the model became slightly more willing to label a field as needing high irrigation. This helped it catch a few more of those important cases without making the overall predictions worse, but the improvement was small. In contrast, I tried building the Multinomial Naive Bayes model for extra practice and as expected it did not perform well overall. While it was able to identify many of the fields that needed high irrigation, it also incorrectly labeled a large number of fields as high when they were not. The tradeoff with the Gaussian Naive Bayes and the Multinomial Naive Bayes model is that they are fast, simple, and easy to implement, but they do not capture complex relationships in the data as well as more advanced models. Naive Bayes is a smart choice when you care about speed, simplicity, and getting a solid baseline quickly, especially when the data roughly fits its assumptions or when perfect accuracy isn’t the top priority.